In [1]:
# ============================================================
# Yukawa Potential Bound State Solver
# Uses the radial Schrödinger equation with Numerov's method
# and a shooting/matching approach to find bound state energies
# ============================================================

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from jwanglibs import rootfinder as rtf

In [3]:
# ------------------------------------------------------------
# Effective potential and Schrödinger equation in Numerov form
# ------------------------------------------------------------

def V_eff(r, L):
    """Centrifugal barrier + Yukawa attraction."""
    return hbar**2 * L*(L+1) / (2*m*r**2) - V0*np.exp(-lamda*r)/r

def f(r, L):
    """Numerov form: u'' + f(r)*u = 0"""
    return 2*m*(E - V_eff(r, L)) / hbar**2

In [4]:
# ------------------------------------------------------------
# Numerov integrator
# ------------------------------------------------------------

def numerov(fr, u_init, n_steps, x0, h):
    """
    4th-order integrator for u'' + f(x)*u = 0.
    fr      : f(x), single-argument callable
    u_init  : [u0, u1], two starting values
    n_steps : steps beyond the initial two points
    x0      : x value at u_init[0]
    h       : step size (negative for rightward-to-leftward integration)
    Returns (u, node_count).
    """
    u = list(u_init)
    nodes = 0
    c  = h**2 / 12.
    f0 = fr(x0)
    f1 = fr(x0 + h)
    x  = x0 + h

    for _ in range(n_steps):
        x  += h
        f2  = fr(x)
        u.append((2*(1 - 5*c*f1)*u[-1] - (1 + c*f0)*u[-2]) / (1 + c*f2))
        if u[-1] * u[-2] < 0.0:
            nodes += 1
        f0, f1 = f1, f2

    return u, nodes

In [5]:
# ------------------------------------------------------------
# Shooting function
# ------------------------------------------------------------

def shoot(En, L):
    """
    Integrate inward from xL and outward from xR, return
    the Wronskian mismatch at the matching point xm.
    A bound state exists where shoot(E, L) = 0.
    """
    global E
    E  = En
    c  = h**2 / 12.
    fr = lambda r: f(r, L)
    xm = xL + M*h

    wfup, _ = numerov(fr, [0., 0.001], M,     xL,  h)
    wfdn, _ = numerov(fr, [0., 0.001], N - M, xR, -h)

    dup = ((1 + c*fr(xm+h))*wfup[-1] - (1 + c*fr(xm-h))*wfup[-3]) / (2*h)
    ddn = ((1 + c*fr(xm+h))*wfdn[-3] - (1 + c*fr(xm-h))*wfdn[-1]) / (2*h)

    return dup*wfdn[-2] - wfup[-2]*ddn

In [6]:
# ------------------------------------------------------------
# Parameters (dimensionless units: hbar = m = 1)
# ------------------------------------------------------------

xL, xR = 1e-5, 15.
N       = 2200
m       = 1.0
hbar    = 1.0
V0      = 1.0      # Yukawa strength
lamda   = 1.0      # inverse range
h       = (xR - xL) / N
r_grid  = np.linspace(xL, xR, N + 1)

Lmax    = 4
M       = 100      # matching point index
dE      = 0.001

In [8]:
# ------------------------------------------------------------
# Eigenvalue search
# ------------------------------------------------------------

EL   = []   # bound state energies per L
PsiL = []   # normalized wavefunctions per L

for L in range(Lmax):
    Ea, PsiA = [], []
    E1 = -0.5 / (L+1)**2 - 0.5   # start sweep below expected ground state

    while E1 < -4*dE:
        E1 += dE
        if shoot(E1, L) * shoot(E1 + dE, L) >= 0:
            continue

        E = rtf.bisect(lambda En: shoot(En, L), E1, E1 + dE, 1e-8)

        # Build and normalize wavefunction at this energy
        fr = lambda r: f(r, L)
        wfup, nup = numerov(fr, [0., 0.001], M,     xL,  h)
        wfdn, ndn = numerov(fr, [0., 0.001], N - M, xR, -h)

        scale    = wfup[-1] / wfdn[-1]
        wfdn_arr = np.array(wfdn[::-1]) * scale
        psi      = np.concatenate((wfup[:-1], wfdn_arr))
        psi     /= np.sqrt(np.trapezoid(psi**2, r_grid))   # normalize

        n = L + 1 + len(Ea)
        print(f"  n={n}  l={L}  nodes={nup+ndn}  E={E:.6f}")
        Ea.append(E)
        PsiA.append(psi)

    EL.append(Ea)
    PsiL.append(PsiA)

ValueError: operands could not be broadcast together with shapes (2200,) (2202,) 

In [ ]:
# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(fr'Yukawa Potential  ($V_0 = {V0}$, $\lambda = {lamda}$)', fontsize=13)

# 1. Energy level diagram
ax = axes[0]
colors = ['#2166ac', '#d6604d', '#1a9850', '#8073ac']
for L in range(Lmax):
    for i, En in enumerate(EL[L]):
        ax.plot([L - 0.35, L + 0.35], [En, En],
                color=colors[L], lw=2,
                label=f'$\ell$={L}' if i == 0 else '')
ax.axhline(0, color='k', lw=0.8, ls='--', label='E=0 (continuum)')
ax.set_xlabel(r'$\ell$', fontsize=12)
ax.set_ylabel(r'$E$',    fontsize=12)
ax.set_xticks(range(Lmax))
ax.set_title('Bound State Energies')
ax.legend(fontsize=9)

# 2. Radial wavefunctions u(r) for L=0
ax2 = axes[1]
for i, psi in enumerate(PsiL[0]):
    ax2.plot(r_grid, psi, lw=1.8, label=f'n={i+1},  E={EL[0][i]:.4f}')
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_xlim(0, 12)
ax2.set_xlabel(r'$r$',            fontsize=12)
ax2.set_ylabel(r'$u(r) = rR(r)$', fontsize=12)
ax2.set_title(r'Radial Wavefunctions  ($\ell = 0$)')
ax2.legend(fontsize=9)

# 3. Probability density |u(r)|² for L=0
ax3 = axes[2]
for i, psi in enumerate(PsiL[0]):
    ax3.plot(r_grid, psi**2, lw=1.8, label=f'n={i+1},  E={EL[0][i]:.4f}')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax3.set_xlim(0, 12)
ax3.set_xlabel(r'$r$',       fontsize=12)
ax3.set_ylabel(r'$|u(r)|^2$', fontsize=12)
ax3.set_title(r'Probability Density  ($\ell = 0$)')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()